Click "Copy to Drive" above to copy this file to your Colab account.


# Bonus: Neural Network Architecture Visualization with ANN_Viz

This bonus demonstration trains a feedforward neural network on the Graduate Admissions (`GRE.csv`) dataset and uses `ann_visualizer` to generate and display an architectural diagram showing input features, hidden nodes, connections, and output activation.

In [ ]:
# Install required visualization packages
!pip install ann_visualizer graphviz

In [ ]:
import types
import os
import pandas as pd
import numpy as np
import tensorflow as tf
import keras
import keras.layers
from keras.models import Sequential
from keras.layers import Dense, Input
import graphviz

# Ensure compatibility with ann_visualizer's legacy Keras 2 layer checks in modern Keras 3
if not hasattr(keras.layers, 'core'):
    core = types.ModuleType('core')
    core.Dense = keras.layers.Dense
    core.Dropout = keras.layers.Dropout
    core.Flatten = keras.layers.Flatten
    keras.layers.core = core

In [ ]:
# Load GRE Admissions dataset directly from course repository
url = "https://raw.githubusercontent.com/DataAnalytics808/DASC-522-demo-repository/main/data/GRE.csv"
df = pd.read_csv(url)
print("Dataset preview:")
print(df.head())

# Features: gre, gpa, rank; Target: admit
X = df[['gre', 'gpa', 'rank']].values
y = df['admit'].values

# Standardize inputs for stable gradient descent
mean = X.mean(axis=0)
std = X.std(axis=0)
X_scaled = (X - mean) / std

In [ ]:
# Define neural network architecture
model = Sequential([
    Input(shape=(3,)),
    Dense(6, activation='relu', name='hidden_1'),
    Dense(4, activation='relu', name='hidden_2'),
    Dense(1, activation='sigmoid', name='output')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

# Train the model
history = model.fit(X_scaled, y, epochs=25, batch_size=16, verbose=1)

In [ ]:
# Prepare layer shape metadata required by ann_visualizer in modern Keras 3
for i, layer in enumerate(model.layers):
    if i == 0:
        layer.input_shape = (None, 3)
    layer.output_shape = (None, layer.units)

# Import ann_viz and generate visualization
from ann_visualizer.visualize import ann_viz

ann_viz(model, title="Graduate Admissions Neural Network", filename="admissions_nn")
print("ANN_Viz successfully generated 'admissions_nn' (DOT graph) and 'admissions_nn.pdf'!")

In [ ]:
# Display visualization inline in notebook
display_graph = graphviz.Source.from_file('admissions_nn')
display_graph